# EDA — Duración de los viajes de Citi Bike

**Curso:** Aprendizaje Automático en la nube
**Datos:** Citi Bike, Jersey City y Hoboken — julio de 2026 (109.095 viajes)

---

## La pregunta del proyecto

> **¿Cuánto va a durar un viaje en bicicleta?**

Es un problema de **regresión**: la variable a predecir es continua (minutos).

Este EDA no busca construir el modelo, sino responder lo que hay que saber antes de intentarlo:
cómo se distribuye la duración, qué variables la explican, qué datos son confiables
y qué trampas tiene el dataset.

## Nota sobre el origen de los datos

Cada fila es un viaje registrado por el sistema. **No son datos crudos**: Citi Bike aplica
filtros antes de publicarlos. Lo comprobamos en la sección de calidad, porque condiciona
lo que podemos concluir.

Un punto importante de encuadre: el archivo no trae una columna de duración. Hay que
**construirla** a partir de las marcas de tiempo. Lo mismo ocurre con la distancia, la hora
y el día de la semana. Esa construcción de variables es parte del análisis, no un trámite previo.

---

## 1. Preparación

Las librerías de análisis y, sobre todo, **el paquete del proyecto**: la ruta del
archivo, los nombres de columnas y la lógica de lectura no viven en el notebook,
viven en `src/trips/`. El notebook explica; el paquete ejecuta. Así el mismo
código sirve luego para el script de entrenamiento, sin copiar y pegar.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from trips.config import TARGET_COLUMN
from trips.data.clean import clean_trips
from trips.data.load import describe_raw, load_trips

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.autolayout"] = True

AZUL, ROJO, GRIS = "#4C78A8", "#E45756", "#8C8C8C"
PALETA_USUARIO = {"member": AZUL, "casual": ROJO}
PALETA_BICI = {"classic_bike": GRIS, "electric_bike": "#54A24B"}

## 2. Carga de los datos

`load_trips()` hace tres cosas y ninguna más:

1. **Lee** el CSV desde la ruta declarada en `config.py` (configurable con la
   variable de entorno `TRIPS_RAW_DATA`, para que cada integrante apunte a su copia).
2. **Tipa** las columnas al leer: identificadores como texto (para no perder ceros
   a la izquierda), `rideable_type` y `member_casual` como categorías (memoria),
   y `started_at`/`ended_at` como fechas.
3. **Valida el esquema**: si el archivo no trae las 13 columnas esperadas, falla
   de inmediato. Enterarse aquí es barato; enterarse al entrenar, no.

Lo que **no** hace: limpiar, filtrar ni crear variables. `data/raw` es intocable.


In [ ]:
df = load_trips()

print(f"{df.shape[0]:,} viajes x {df.shape[1]} columnas")
df.head()

### Ficha técnica del archivo crudo

Antes de graficar nada: cuánto pesa, qué periodo cubre, si hay duplicados y
dónde faltan datos. Estas cinco líneas deciden buena parte del trabajo de limpieza.


In [ ]:
describe_raw(df)

### Tipos y estadísticos de partida

`info()` confirma que los tipos declarados llegaron bien. `describe()` sobre las
numéricas es el primer olfateo de valores imposibles (coordenadas en cero,
por ejemplo).


In [ ]:
df.info()

In [ ]:
df.describe(include="number").T

---

## 3. Calidad de los datos

Tres preguntas antes de limpiar nada: **qué son los nulos**, **cómo se ve la duración**
y **qué viajes no son viajes**. Cada respuesta se convierte después en una regla de
limpieza — y ninguna regla entra sin evidencia que la respalde.


### 3.1 Los nulos: ¿qué les falta exactamente?

`describe_raw` mostró que los nulos están todos del lado del final del viaje, pero con
conteos distintos entre columnas (330, 329, 299). Eso significa que **no hay un solo
patrón**. Cruzamos las cuatro columnas para ver cuáles son:


In [ ]:
COLS_FIN = ["end_station_id", "end_station_name", "end_lat", "end_lng"]

patron = df[COLS_FIN].isna().astype(int)
combos = patron.value_counts().reset_index(name="viajes")
combos[combos["viajes"] > 0].sort_values("viajes", ascending=False)

La hipótesis inicial era que fueran bicicletas eléctricas dejadas fuera de estación
(tendrían coordenadas pero no estación). **Es falsa**: solo 1 viaje encaja en ese patrón.

Lo que hay de verdad son 298 viajes a los que les falta *todo* el destino y 31 con nombre
de estación pero sin identificador ni coordenadas. No es un aparcamiento libre: es una
devolución que el sistema no registró bien.

No se reparten al azar — y eso importa para no borrarlos en silencio:


In [ ]:
incompletos = df[df[COLS_FIN].isna().any(axis=1)]

for columna in ["rideable_type", "member_casual"]:
    tabla = pd.DataFrame(
        {
            "incompletos": incompletos[columna].value_counts(),
            "total": df[columna].value_counts(),
        }
    ).assign(pct=lambda d: (d.incompletos / d.total * 100).round(2))
    print(tabla, "\n")

Las eléctricas fallan cuatro veces más que las clásicas (0,43% contra 0,10%) y los
usuarios casuales casi el triple que los miembros (0,57% contra 0,20%). Son 330 filas —
el 0,3% — así que eliminarlas es asumible, pero **hay que dejarlo dicho**: al hacerlo
perdemos algo más de eléctricas y de casuales que del resto.


### 3.2 La duración: dónde están los valores imposibles

Construimos la variable objetivo sobre los datos crudos, solo para mirarla. Todavía no
limpiamos nada — primero hay que saber qué estamos limpiando.


In [ ]:
duracion_cruda = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60

print("Percentiles (minutos)")
for q in [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999, 1.0]:
    print(f"  p{q * 100:>6.1f}   {duracion_cruda.quantile(q):>10.2f}")

print("\nCasos sospechosos")
for etiqueta, mascara in [
    ("duración negativa", duracion_cruda < 0),
    ("menos de 1 minuto", duracion_cruda < 1),
    ("más de 1 hora", duracion_cruda > 60),
    ("más de 24 horas", duracion_cruda > 1440),
]:
    n = int(mascara.sum())
    print(f"  {etiqueta:<20} {n:>7,}  ({n / len(duracion_cruda):.3%})")

Dos lecturas de esta tabla.

**El piso es sospechosamente limpio.** Cero duraciones negativas, cero por debajo de un
minuto, y el mínimo exactamente en 1,00. En datos crudos de verdad eso no pasa: siempre
hay errores de captura. Esta es la prueba de lo que anunciamos en la introducción —
**Citi Bike filtra antes de publicar** y descarta los viajes de menos de 60 segundos. No
tenemos datos crudos, tenemos datos ya depurados por el proveedor, y conviene decirlo
porque condiciona lo que podemos concluir.

**El problema está arriba.** La mediana es de 6,4 minutos, pero el máximo llega a 1.500
(25 horas). La distribución tiene una cola derecha larguísima: entre el p99 y el p100 se
pasa de 65 minutos a 25 horas. Los 44 viajes de más de un día no son paseos largos: son
bicicletas que no volvieron.


### 3.3 Los viajes que no son viajes

Sacar una bicicleta, notar que está dañada y devolverla al mismo sitio produce un
"viaje" de un par de minutos que empieza y termina en la misma estación. Hay que
distinguirlos de los paseos de ida y vuelta, que sí son reales:


In [ ]:
misma_estacion = df["start_station_name"] == df["end_station_name"]

print(f"Ida y vuelta a la misma estación:        {int(misma_estacion.sum()):>7,}")
print(
    f"  de esos, con menos de 2 minutos:       {int((misma_estacion & (duracion_cruda < 2)).sum()):>7,}"
)
print(
    f"  duración mediana de los demás:         {duracion_cruda[misma_estacion & (duracion_cruda >= 2)].median():>7.1f} min"
)

4.480 viajes vuelven a la estación de origen, y su duración mediana es perfectamente
normal: son paseos. Solo 278 duran menos de dos minutos, y esos sí son intentos fallidos.
La regla tiene que ser específica — **misma estación _y_ menos de dos minutos** — porque
cualquiera de las dos condiciones por separado se llevaría por delante viajes legítimos.


---

## 4. Limpieza

Las decisiones anteriores viven en `src/trips/data/clean.py`, y los umbrales en
`config.py`. El notebook no reimplementa nada: importa la misma función que corre el
script, así que lo que ves aquí es exactamente lo que produce `make data`.

Un paso construye y tres eliminan:

| Paso | Qué hace | Por qué |
|---|---|---|
| 1 | Construye `duracion_min` | El archivo no la trae |
| 2 | Elimina viajes sin destino | No sabemos dónde terminaron (330) |
| 3 | Elimina duraciones > 24 h | Una bici más de un día fuera no volvió |
| 4 | Elimina falsos viajes | Misma estación y < 2 min (278) |


In [ ]:
df_limpio = clean_trips(df)

**El paso 3 elimina cero filas, y aun así el máximo baja de 1.500 a 1.382 minutos.**

No es un error: los 44 viajes de más de un día estaban *todos* dentro de las 330 filas sin
destino, así que el paso 2 ya se los había llevado. Las dos anomalías que parecían
independientes son el mismo fenómeno visto por dos lados — una bicicleta que no se devolvió
bien no tiene estación de destino y además acumula una duración absurda.

Dejamos la regla igual, como barrera de seguridad: si el mes que viene aparece un viaje de
30 horas *con* destino registrado, el paso 3 lo atrapa. Una regla que no dispara hoy no es
una regla inútil, es una que todavía no ha hecho falta.


In [ ]:
print(f"Crudo:  {len(df):,} viajes")
print(f"Limpio: {len(df_limpio):,} viajes  ({len(df_limpio) / len(df):.2%})\n")

df_limpio[TARGET_COLUMN].describe().round(2)

Quedan 108.487 viajes, el 99,44% del archivo original. Medio punto porcentual de datos
a cambio de que cada fila sea un viaje real.

El resultado está guardado en `data/processed/viajes_limpio.parquet` (lo escribe el script,
no este notebook: los notebooks explican, el paquete ejecuta).

> **Siguiente parte:** el EDA propiamente dicho sobre los datos limpios — la distribución de
> la duración, las variables derivadas (hora del día, día de la semana, distancia entre
> estaciones) y qué relación tiene cada una con lo que queremos predecir.
